# Generalization: the same two extensions transfer to fuel-cell gas-diffusion layers

This notebook tests whether the framework is battery-specific or a transferable recipe. The manuscript argues the closure applies wherever conduction through a porous solid is set by **rarefied pore gas + a process-dependent contact network**. PEM fuel-cell gas-diffusion layers (GDLs) are the strongest test of that claim: a fibrous carbon medium, compressed in the cell (compaction plays the role of calendering's $\Pi$), with carbon-black/PTFE microporous layers (MPLs) whose ~100 nm pores invite a Knudsen term. Crucially the GDL thermal literature reached the *same* qualitative conclusion independently -- "the contact region between fibers is important for the thermal conductivity of a GDL" (Pfrang et al., IntechOpen, compiling Burheim et al. 2011).

We use published through-plane conductivities of dry GDLs versus compaction load (Burheim et al. 2011, via the IntechOpen compilation, `data/raw/gdl_thermal_literature.csv`) and ask three questions, each the GDL analogue of an electrode finding:
1. Does the point-contact closure ($\varphi=0$) **underpredict**, as it did for electrodes and separators?
2. Does a **contact fraction that rises with compaction** reproduce the measured $k$-vs-load trend, as $\varphi(\Pi)$ did for calendering?
3. Is **Knudsen negligible for the micrometre GDL pores but decisive for the 100 nm MPL**, as it was small for electrodes but large for separators?

In [1]:
import numpy as np, pandas as pd
try:
    import electrode_thermal as et          # installed package (pip install -e .)
except ImportError:
    import sys; sys.path.insert(0, "src"); import electrode_thermal as et
from electrode_thermal import (lambda_eff_coating, lambda_eff_coating_contact,
                               knudsen_number, pore_diameter, mean_free_path, LAMBDA_AIR)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import os; os.makedirs("figures/generalization", exist_ok=True)

gdl = pd.read_csv("data/raw/gdl_thermal_literature.csv")
# GDL material model: carbon fibre lam_s (through-plane effective), air pore gas, fibre d~9 um
LAM_S_CARBON = 120.0      # carbon fibre (manufacturer value used in the GDL literature)
D_FIBRE = 9e-6
print(gdl.to_string(index=False))

               gdl  ptfe_pct  compression_bar state  k_through_W_mK  porosity                                                            source
   Toray TGP-H-060         0              4.6   dry            0.41      0.78 Burheim et al. 2011 (via Pfrang/IntechOpen Table 2); low-load end
   Toray TGP-H-060         0             13.9   dry            0.66      0.78      Burheim et al. 2011; high-load end (k rises with compaction)
  E-Tek EC-CC1-060         0              4.6   dry            0.28      0.80                                 Burheim et al. 2011; low-load end
  E-Tek EC-CC1-060         0             13.9   dry            0.32      0.80                                Burheim et al. 2011; high-load end
SGL Sigracet 10 AA         5              4.6   dry            0.30      0.80                                 Burheim et al. 2011; low-load end
SGL Sigracet 10 AA         5             13.9   dry            0.42      0.80                                Burheim et al. 2011; high-l

## 1. Point-contact closure underpredicts GDLs (same signature as electrodes/separators)

In [2]:
print("zero-fit (phi=0) ZBS+Knudsen vs measured, dry GDLs:")
print(f"{'gdl':22}{'eps':>6}{'meas':>8}{'phi=0 pred':>11}")
for r in gdl.drop_duplicates('gdl').itertuples():
    pred = float(lambda_eff_coating(r.porosity, D_FIBRE, LAM_S_CARBON, LAMBDA_AIR, True))
    sub = gdl[gdl.gdl==r.gdl]
    print(f"{r.gdl:22}{r.porosity:6.2f}{sub.k_through_W_mK.min():.2f}-{sub.k_through_W_mK.max():.2f}{pred:11.3f}")
print("\n-> phi=0 underpredicts by ~6-8x: point contact ignores fibre-fibre conduction,")
print("   exactly as it underpredicted electrode coatings and separators (a fibrous medium,")
print("   so like separators the sphere-pack base geometry is itself approximate).")

zero-fit (phi=0) ZBS+Knudsen vs measured, dry GDLs:
gdl                      eps    meas phi=0 pred


Toray TGP-H-060         0.780.41-0.66      0.086
E-Tek EC-CC1-060        0.800.28-0.32      0.077
SGL Sigracet 10 AA      0.800.30-0.42      0.077

-> phi=0 underpredicts by ~6-8x: point contact ignores fibre-fibre conduction,
   exactly as it underpredicted electrode coatings and separators (a fibrous medium,
   so like separators the sphere-pack base geometry is itself approximate).


**Section 1 result** -- PLACEHOLDER

## 2. A compaction-dependent contact fraction reproduces the k-vs-load trend

For each GDL we solve for the contact fraction $\varphi$ at the low- and high-load points (carbon bridges, $\lambda_b=\lambda_s$). If the mechanism transfers, $\varphi$ should be small (sub-percent, like the VDI sphere value and our electrode $\varphi_0$) and should **increase with compaction load** -- more fibre-fibre contact under pressure, the GDL analogue of calendering.

In [3]:
from scipy.optimize import brentq
def phi_for(k_meas, eps):
    def f(phi):
        return float(lambda_eff_coating_contact(eps, D_FIBRE, LAM_S_CARBON, LAMBDA_AIR, True,
                     phi=max(phi,0.0), lambda_bridge=LAM_S_CARBON)) - k_meas
    return brentq(f, 0.0, 0.05)

print(f"{'gdl':22}{'load[bar]':>10}{'k_meas':>8}{'phi_fit':>9}")
rows=[]
for r in gdl.itertuples():
    phi = phi_for(r.k_through_W_mK, r.porosity)
    rows.append((r.gdl, r.compression_bar, r.k_through_W_mK, phi))
    print(f"{r.gdl:22}{r.compression_bar:10.1f}{r.k_through_W_mK:8.2f}{phi:9.4f}")
res = pd.DataFrame(rows, columns=["gdl","bar","k","phi"])
print("\nphi rises with load for every GDL:")
for g in res.gdl.unique():
    s=res[res.gdl==g].sort_values("bar")
    d=s.phi.iloc[-1]-s.phi.iloc[0]
    print(f"  {g:22} phi {s.phi.iloc[0]:.4f} -> {s.phi.iloc[-1]:.4f}  (delta {d:+.4f} over {s.bar.iloc[0]}-{s.bar.iloc[-1]} bar)")

fig,ax=plt.subplots(figsize=(5,3.4))
for g in res.gdl.unique():
    s=res[res.gdl==g].sort_values("bar"); ax.plot(s.bar,s.phi,"o-",label=g,ms=5)
ax.set_xlabel("compaction load [bar]"); ax.set_ylabel("fitted contact fraction phi")
ax.set_title("Contact fraction rises with GDL compaction\n(the phi(Pi) mechanism, transferred)")
ax.axhspan(0.005,0.017,alpha=0.12,color="green")
ax.text(5,0.015,"electrode phi0 band",fontsize=7,color="green")
ax.legend(fontsize=7); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig("figures/generalization/gdl_contact.pdf"); plt.show()

gdl                    load[bar]  k_meas  phi_fit
Toray TGP-H-060              4.6    0.41   0.0058
Toray TGP-H-060             13.9    0.66   0.0102
E-Tek EC-CC1-060             4.6    0.28   0.0038
E-Tek EC-CC1-060            13.9    0.32   0.0045
SGL Sigracet 10 AA           4.6    0.30   0.0042
SGL Sigracet 10 AA          13.9    0.42   0.0064

phi rises with load for every GDL:
  Toray TGP-H-060        phi 0.0058 -> 0.0102  (delta +0.0044 over 4.6-13.9 bar)
  E-Tek EC-CC1-060       phi 0.0038 -> 0.0045  (delta +0.0007 over 4.6-13.9 bar)
  SGL Sigracet 10 AA     phi 0.0042 -> 0.0064  (delta +0.0022 over 4.6-13.9 bar)


C:\Users\StoerkJulius\AppData\Local\Temp\ipykernel_10188\4155431449.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig("figures/generalization/gdl_contact.pdf"); plt.show()


**Section 2 result** -- PLACEHOLDER

## 3. Knudsen: negligible for the GDL paper, decisive for the 100 nm MPL

In [4]:
mfp=float(mean_free_path())
dp_gdl=float(pore_diameter(0.80,D_FIBRE))
kn_gdl=float(knudsen_number(0.80,D_FIBRE))
kn_mpl=mfp/100e-9
print(f"GDL carbon paper: d_pore={dp_gdl*1e6:.1f} um, Kn={kn_gdl:.4f} -> gas retained {100/(1+2*1.64*kn_gdl):.0f}% (negligible)")
print(f"MPL (100 nm pores): Kn={kn_mpl:.2f} -> gas retained {100/(1+2*1.64*kn_mpl):.0f}% (decisive)")
print("\n-> exactly the electrode (small Knudsen) vs separator (large Knudsen) split,")
print("   reappearing within a single fuel-cell component: the fibre paper needs no")
print("   correction, the 100 nm microporous layer loses ~70% of its pore-gas conduction.")

GDL carbon paper: d_pore=24.0 um, Kn=0.0028 -> gas retained 99% (negligible)
MPL (100 nm pores): Kn=0.67 -> gas retained 31% (decisive)

-> exactly the electrode (small Knudsen) vs separator (large Knudsen) split,
   reappearing within a single fuel-cell component: the fibre paper needs no
   correction, the 100 nm microporous layer loses ~70% of its pore-gas conduction.


**Section 3 result** -- PLACEHOLDER

## 4. Summary -- PLACEHOLDER